# Capital One Mobile · VoC Data Collection & Categorization

**Purpose:** Collect Google Play reviews for Capital One Mobile app and categorize them by complaint type for downstream Tableau analysis.

**Output:** `cap1_android_categorized.csv` — 29,101 reviews with category labels, ready for Tableau aggregation.

**Author:** Julia Korneva  
**Data source:** Google Play Store, US locale, English  
**Period:** Last 12 months

## 1. Setup

In [ ]:
!pip install google-play-scraper -q

from google_play_scraper import reviews_all, Sort
import pandas as pd
import re

## 2. Data Collection

Pulling all available reviews for Capital One Mobile from Google Play Store (US, English).  
Note: iOS App Store excluded from v1 — Android captures the majority of public review volume.

In [ ]:
APP_ID = 'com.konylabs.capitalone'

result = reviews_all(
    APP_ID,
    sleep_milliseconds=1000,
    lang='en',
    country='us',
    sort=Sort.NEWEST
)

df = pd.DataFrame(result)
df['at'] = pd.to_datetime(df['at'])

# Filter to last 12 months
start_date = pd.Timestamp.now() - pd.DateOffset(months=12)
df = df[df['at'] >= start_date].reset_index(drop=True)

print(f"Reviews collected: {len(df):,}")
print(f"Date range: {df['at'].min().date()} → {df['at'].max().date()}")
print(f"Avg score: {df['score'].mean():.2f}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Save raw data — collection takes ~10 min, no need to repeat
df.to_csv('cap1_android_raw.csv', index=False)
print(f"Saved: cap1_android_raw.csv ({len(df):,} rows)")

## 3. Exploratory Analysis

In [ ]:
print("=== SCORE DISTRIBUTION ===")
print(df['score'].value_counts().sort_index())
print(f"% Negative (1-2★): {(df['score']<=2).mean()*100:.1f}%")
print(f"% Positive (4-5★): {(df['score']>=4).mean()*100:.1f}%")

print("\n=== MONTHLY VOLUME ===")
print(df.groupby(df['at'].dt.to_period('M')).size())

print("\n=== APP VERSION COVERAGE ===")
print(f"Reviews with version tag: {df['reviewCreatedVersion'].notna().sum():,} ({df['reviewCreatedVersion'].notna().mean()*100:.1f}%)")
print(f"Unique versions: {df['reviewCreatedVersion'].nunique()}")

print("\n=== CAPITAL ONE REPLY RATE ===")
print(f"Reviews with reply: {df['replyContent'].notna().sum()} ({df['replyContent'].notna().mean()*100:.2f}%)")

## 4. Complaint Categorization

**Method:** Keyword-based rule matching — interpretable and auditable without ML complexity for v1.  
Categories derived from manual review of top keywords in 1-2★ reviews.  
Rule: first matching category wins. Returns `Other` if no match.

In [ ]:
CATEGORIES = {
    'App Stability': [
        'crash', 'crashes', 'crashing', 'freeze', 'froze', 'frozen',
        'glitch', 'glitchy', 'bug', 'buggy', 'broken',
        "won't load", 'wont load', 'not load', 'not loading',
        "won't open", 'wont open', "doesn't open", 'doesnt open',
        'slow', 'lag', 'laggy',
        "doesn't work", 'doesnt work', 'does not work', 'not working',
        'keeps stopping', 'keeps crashing', 'keeps shutting',
        'unusable', 'never functions', 'never works',
        'force close', 'force closes',
        'update', 'updated', 'after update', 'latest version',
        'reinstalled', 'reinstall', 'uninstall',
    ],
    'Login & Access': [
        'log in', 'login', 'log-in', 'logging in',
        'sign in', 'sign-in', 'signin', 'signing in',
        'password', 'passcode',
        'fingerprint', 'face id', 'faceid', 'biometric', 'touch id',
        'locked out', 'lock out', 'locked me out',
        "can't access", 'cant access', 'cannot access',
        'two factor', '2fa', 'verification code', 'authenticate',
    ],
    'Payments & Transfers': [
        'zelle', 'transfer', 'transferring', 'transferred',
        'payment', 'pay bill', 'paid bill', 'paying',
        'ach', 'wire', 'send money', 'received money',
        'bill pay', 'autopay', 'auto pay', 'recurring',
        'deposit', 'mobile deposit', 'check deposit',
    ],
    'Card Issues': [
        'declined', 'card decline', 'debit card', 'credit card',
        'card not work', 'chip', 'tap to pay', 'apple pay', 'google pay',
        'virtual card', 'replace card', 'new card', 'lost card',
        'fraud', 'unauthorized', 'unlock my card', 'card frozen',
    ],
    'Account Management': [
        'close account', 'closed account', 'open account',
        'balance', 'statement', 'overdraft', 'fee', 'fees',
        'interest rate', 'savings', 'checking account',
        'unable to', 'cannot', 'add bank', 'link account',
    ],
    'UX & Navigation': [
        'menu', 'navigate', 'navigation', 'interface', 'ui', 'ux',
        'confusing', 'hard to find', 'hard to use', 'not intuitive',
        'cluttered', 'clunky', 'layout', 'button',
    ],
    'Ads & Promotions': [
        'ads', 'advertisement', 'pop up', 'popup', 'pop-up',
        'promotion', 'promo', 'upsell', 'spam', 'marketing',
    ],
    'Customer Service': [
        'customer service', 'customer support', 'agent', 'representative',
        'on hold', 'phone call', 'no help', 'no response', 'rude',
    ],
}

def categorize(text):
    """Assign first matching category. Returns 'Other' if no match."""
    if pd.isna(text):
        return 'Other'
    text_lower = str(text).lower()
    for category, keywords in CATEGORIES.items():
        for kw in keywords:
            if kw in text_lower:
                return category
    return 'Other'

df['category'] = df['content'].apply(categorize)

## 5. Coverage Check

In [ ]:
print("=== ALL REVIEWS ===")
print(df['category'].value_counts())
print(f"\nOther %: {(df['category']=='Other').mean()*100:.1f}%")

print("\n=== NEGATIVE REVIEWS (1-2★) — pain points ===")
neg = df[df['score'] <= 2]
print(neg['category'].value_counts())
print(f"\nOther in negatives: {(neg['category']=='Other').mean()*100:.1f}%")

## 6. Export

In [ ]:
df.to_csv('cap1_android_categorized.csv', index=False)
print(f"Saved: {len(df):,} rows, {df['category'].nunique()} categories")